In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_MOVES = "grandmasters.silver.game_move_events"

DEVELOPMENT_MODE = True

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
display(bronze_df)

In [0]:
# ============================================================
# EXTRACT GAME EVENTS
#
# Purpose:
# Parse 'featured' events from Bronze into structured columns.
# One record represents one chess game.
# ============================================================

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

featured_schema = StructType([
    StructField("t", StringType()),
    StructField("d", StructType([
        StructField("id", StringType()),
        StructField("orientation", StringType()),
        StructField("fen", StringType()),
        StructField("winner", StringType()),
        StructField("players", ArrayType(
            StructType([
                StructField("color", StringType()),
                StructField("rating", IntegerType()),
                StructField("seconds", IntegerType()),
                StructField("user", StructType([
                    StructField("name", StringType()),
                    StructField("id", StringType())
                ]))
            ])
        ))
    ]))
])

game_events_df = (
    bronze_df
    .filter(col("event_type") == "featured")
    .withColumn(
        "json",
        from_json(col("raw_event"), featured_schema)
    )
    .select(
        col("game_id"),
        col("ingested_at"),

        col("json.d.players")[0]["user"]["name"].alias("white_player"),
        col("json.d.players")[1]["user"]["name"].alias("black_player"),

        col("json.d.players")[0]["rating"].alias("white_rating"),
        col("json.d.players")[1]["rating"].alias("black_rating"),

        col("json.d.players")[0]["seconds"].alias("time_control_seconds"),

        col("json.d.winner").alias("winner"),

        col("json.d.fen").alias("starting_fen")
    )
)

display(game_events_df)

In [0]:
# ============================================================
# EXTRACT MOVE EVENTS
#
# Purpose:
# Parse every FEN event into a structured move timeline.
# One row represents one move in a chess game.
# ============================================================

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

fen_schema = StructType([
    StructField("t", StringType()),
    StructField("d", StructType([
        StructField("fen", StringType()),
        StructField("lm", StringType()),
        StructField("wc", IntegerType()),
        StructField("bc", IntegerType())
    ]))
])

move_events_df = (
    bronze_df
    .filter(col("event_type") == "fen")
    .withColumn(
        "json",
        from_json(col("raw_event"), fen_schema)
    )
    .select(
        "game_id",
        "move_seq",
        "ingested_at",

        col("json.d.fen").alias("fen"),
        col("json.d.lm").alias("last_move"),

        col("json.d.wc").alias("white_clock_secs"),
        col("json.d.bc").alias("black_clock_secs")
    )
)

display(move_events_df)

In [0]:
# ============================================================
# VALIDATE MOVE EVENTS
# ============================================================

from pyspark.sql import functions as F

print(f"Total Moves : {move_events_df.count()}")

print("\nDuplicate (game_id, move_seq) Check")

display(
    move_events_df
    .groupBy("game_id", "move_seq")
    .count()
    .filter("count > 1")
)

print("\nMissing Values")

display(
    move_events_df.select(
        *[
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in move_events_df.columns
        ]
    )
)

print("\nMoves Per Game")

display(
    move_events_df
    .groupBy("game_id")
    .agg(
        F.max("move_seq").alias("max_move_seq"),
        F.count("*").alias("rows")
    )
)

In [0]:
# ============================================================
# WRITE SILVER MOVE EVENTS
# ============================================================

( 
    move_events_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_MOVES)
)

print(f"Successfully wrote {move_events_df.count()} rows to {SILVER_MOVES}")